# Modelo de Clasificación mediante K-Nearest Neighbors (KNN) para la predicción de niveles de TCH y Sacarosa

Este cuaderno tiene como objetivo desarrollar un modelo de clasificación basado en el algoritmo de K-Nearest Neighbors (KNN) para predecir los niveles de productividad de la caña de azúcar, específicamente el TCH (Toneladas de Caña por Hectárea) y el contenido de sacarosa.

Para ello, se utilizan datos históricos suministrados por el Ingenio Providencia, los cuales contienen información agronómica, climática y de manejo del cultivo. Las variables continuas de interés son transformadas en categorías (Bajo, Medio y Alto), permitiendo abordar el problema como una tarea de clasificación multiclase.

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score
)

import matplotlib.pyplot as plt
import sys
import os

sys.path.append(os.path.abspath('..'))

from src.evaluation import evaluar_modelo

## 2. Carga de datos

El dataset `BD_IPSA_1940.xlsx` contiene información histórica sobre variables agronómicas, ambientales y de manejo del cultivo de caña de azúcar. Este conjunto de datos será utilizado para entrenar y evaluar los modelos de clasificación orientados a predecir los niveles de TCH y sacarosa.

In [ ]:
RAW_DATA_PATH = '../data/BD_IPSA_1940.xlsx'

df = pd.read_excel(RAW_DATA_PATH)
df.head()

## 3. Creación de clases para las variables objetivo

Con el fin de abordar el problema como una tarea de clasificación multiclase, las variables continuas `TCH` y `sacarosa` se discretizan en tres categorías: Bajo, Medio y Alto.

Se utiliza `pd.qcut` para generar clases balanceadas mediante cuantiles, lo que permite reducir problemas de desbalance que podrían afectar negativamente el desempeño del modelo.

In [ ]:
df["TCH_clase"] = pd.qcut(
    df["TCH"],
    q=3,
    labels=["Bajo", "Medio", "Alto"]
)

df["sacarosa_clase"] = pd.qcut(
    df["sacarosa"],
    q=3,
    labels=["Bajo", "Medio", "Alto"],
    duplicates="drop"
)

## 4. Definición de variables predictoras

Se seleccionan variables relevantes desde el punto de vista agronómico, relacionadas con el desarrollo del cultivo, las condiciones climáticas, el manejo agrícola y las características del lote.

Estas variables se consideran potencialmente influyentes en el rendimiento y la calidad de la caña, por lo que pueden aportar información valiosa para la clasificación de los niveles de TCH y sacarosa.

In [ ]:
features = [
    "edad", "cortes", "dosismad", "semsmad",
    "lluvias", "pct_diatrea", "vejez",
    "tipocorte", "variedad", "madurada",
    "producto", "mes", "periodo", "grupo_tenencia"
]

## 5. Creación de subconjuntos para TCH y sacarosa

A partir del dataset original, se generan dos subconjuntos independientes: uno para la predicción de `TCH_clase` y otro para la predicción de `sacarosa_clase`.

Esta separación permite trabajar cada problema de clasificación de forma individual, manteniendo el mismo conjunto de variables predictoras, pero cambiando la variable objetivo según el caso de estudio.

In [ ]:
data_tch = df[features + ["TCH_clase"]].dropna(subset=["TCH_clase"]).copy()
data_sac = df[features + ["sacarosa_clase"]].dropna(subset=["sacarosa_clase"]).copy()

## 6. Modelo KNN para TCH

### 6.1 Definición de variables y partición de datos

En esta etapa, se define `TCH_clase` como variable objetivo y se seleccionan las variables predictoras previamente establecidas. Posteriormente, el conjunto de datos se divide en entrenamiento (80%) y prueba (20%).

Se utiliza estratificación mediante `stratify=y` con el fin de conservar la proporción de clases en ambos subconjuntos, lo que permite obtener una evaluación más representativa del desempeño del modelo.

In [ ]:
X = data_tch[features]
y = data_tch["TCH_clase"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### 6.2 Preprocesamiento de datos

Las variables numéricas se imputan con la mediana y posteriormente se escalan mediante `StandardScaler`. Las variables categóricas se imputan con la moda y se codifican a través de `OneHotEncoder`.

Este paso es fundamental en KNN, ya que el modelo se basa en distancias y es altamente sensible a la escala de los datos. Sin una estandarización adecuada, las variables con mayor magnitud podrían dominar el cálculo de distancias y afectar negativamente el desempeño del modelo.

In [ ]:
numericas = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categoricas = X.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numericas),
    ("cat", categorical_transformer, categoricas)
])

### 6.3 Construcción del pipeline

Se construye un pipeline que integra el preprocesamiento de los datos con el modelo KNN en un único flujo de trabajo. Esto garantiza que todas las transformaciones (imputación, escalado y codificación) se apliquen de manera consistente tanto en el entrenamiento como en la evaluación del modelo, evitando fugas de información (data leakage).

In [ ]:
pipe_knn = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier())
])

### 6.4 Búsqueda de hiperparámetros

Se optimizan los principales hiperparámetros del modelo KNN mediante validación cruzada estratificada. En particular, se evalúa el número de vecinos (`k`) y el tipo de ponderación (`weights`).

El número de vecinos controla el nivel de generalización del modelo: valores pequeños pueden generar sobreajuste, mientras que valores grandes pueden suavizar en exceso la frontera de decisión. Por su parte, el parámetro `weights` permite asignar mayor importancia a los vecinos más cercanos, lo cual puede mejorar el desempeño en conjuntos de datos con solapamiento entre clases.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    "model__n_neighbors": list(range(3, 16, 2)),
    "model__weights": ["uniform", "distance"]
}

grid_knn = GridSearchCV(
    pipe_knn,
    param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)

### 6.5 Entrenamiento del modelo

Se entrena el modelo utilizando `GridSearchCV` sobre el conjunto de entrenamiento (`X_train`, `y_train`). Después del entrenamiento, se muestran los mejores hiperparámetros encontrados y el mejor puntaje de F1 macro obtenido durante la validación cruzada.

In [ ]:
grid_knn.fit(X_train, y_train)

best_model_knn = grid_knn.best_estimator_

print("Mejores hiperparámetros:", grid_knn.best_params_)
print("Mejor F1 CV:", grid_knn.best_score_)

El mejor modelo encontrado utiliza `k = 5` vecinos con ponderación por distancia (`weights = distance`). Este resultado sugiere que dar mayor importancia a los vecinos más cercanos mejora la capacidad del modelo para capturar patrones locales en los datos.

El F1 macro obtenido en validación cruzada (≈ 0.48) indica un mejor desempeño en comparación con el modelo de regresión logística, lo cual sugiere que la estructura del problema presenta relaciones no lineales.

### 6.6 Evaluación del modelo

El modelo KNN presenta un F1 macro de aproximadamente 0.49, lo cual representa una mejora frente al modelo de regresión logística. Esto indica que el enfoque basado en vecinos permite capturar mejor la estructura de los datos.

Se observa que las clases "Alto" y "Bajo" presentan un desempeño más equilibrado, mientras que la clase "Medio" sigue siendo la más difícil de clasificar. Este comportamiento es consistente con la presencia de solapamiento entre clases en el espacio de características.

El valor del índice de Cohen’s Kappa (≈ 0.24) indica un acuerdo moderado entre las predicciones del modelo y los valores reales, lo que sugiere que, aunque el modelo mejora respecto a la regresión logística, aún existen limitaciones en la separabilidad de las clases.

In [ ]:
y_pred_knn = best_model_knn.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print("F1 Macro:", f1_score(y_test, y_pred_knn, average="macro"))
print(classification_report(y_test, y_pred_knn))

evaluar_modelo(y_test, y_pred_knn)

La mejora observada en el desempeño del modelo KNN respecto a la regresión logística sugiere que la relación entre las variables predictoras y la variable objetivo no es completamente lineal.

Dado que KNN es un modelo no paramétrico basado en distancias, su capacidad para capturar patrones locales permite una mejor clasificación en escenarios donde existe solapamiento entre clases.

## 7. Modelo KNN para la predicción de sacarosa

El modelo KNN permite capturar patrones locales en los datos, lo cual puede mejorar la clasificación frente a modelos lineales. Si el desempeño es superior al de la regresión logística, esto indicaría que las relaciones entre variables son de tipo no lineal.

### 7.1 Definición de variables y partición de datos

En esta sección, se define `sacarosa_clase` como variable objetivo, manteniendo el mismo conjunto de variables predictoras utilizado previamente. Esto permite comparar de manera consistente el desempeño del modelo entre TCH y sacarosa.

Posteriormente, el conjunto de datos se divide en entrenamiento (80%) y prueba (20%), utilizando estratificación para preservar la proporción de clases.

In [ ]:
X_sac = data_sac[features]
y_sac = data_sac["sacarosa_clase"]

X_train_sac, X_test_sac, y_train_sac, y_test_sac = train_test_split(
    X_sac, y_sac,
    test_size=0.2,
    random_state=42,
    stratify=y_sac
)

### 7.2 Entrenamiento del modelo para la clasificación de sacarosa

Se entrena el modelo KNN utilizando validación cruzada (`GridSearchCV`) sobre el conjunto de entrenamiento, con el objetivo de identificar la mejor combinación de hiperparámetros para la predicción de `sacarosa_clase`.

Se utiliza la misma configuración empleada en el modelo de TCH, lo que permite mantener consistencia metodológica y facilitar la comparación entre ambos resultados.

In [ ]:
grid_knn_sac = GridSearchCV(pipe_knn, param_grid, cv=cv, scoring="f1_macro")

grid_knn_sac.fit(X_train_sac, y_train_sac)

best_knn_sac = grid_knn_sac.best_estimator_

### 7.3 Evaluación del modelo

El modelo entrenado se evalúa sobre el conjunto de prueba utilizando el F1 macro como métrica principal, ya que permite medir el desempeño del modelo considerando todas las clases de forma equilibrada.

Este análisis permite determinar si el modelo KNN logra capturar de manera efectiva los patrones asociados a la variable sacarosa y si presenta mejoras frente a la regresión logística.

In [ ]:
y_pred_sac = best_knn_sac.predict(X_test_sac)

print("F1:", f1_score(y_test_sac, y_pred_sac, average="macro"))

evaluar_modelo(y_test_sac, y_pred_sac)

El modelo KNN obtuvo un F1 macro cercano a 0.50 en el conjunto de prueba, lo cual indica un desempeño moderado en la clasificación de las tres categorías de sacarosa.

Este resultado es ligeramente superior al obtenido en la predicción de TCH, lo que sugiere que la variable sacarosa presenta una mejor separabilidad entre sus clases. Esto puede estar relacionado con una mayor consistencia en los factores que influyen en la calidad del cultivo frente al rendimiento.

El índice de Cohen’s Kappa (≈ 0.26) indica un nivel de acuerdo moderado entre las predicciones del modelo y los valores reales. Esto evidencia que, aunque el modelo logra capturar parte de la estructura de los datos, aún existen limitaciones en la capacidad de clasificación.

Al analizar la matriz de confusión, se observa que la clase "Medio" continúa siendo la más difícil de predecir, presentando confusión tanto con la clase "Bajo" como con la clase "Alto". Este comportamiento sugiere un alto grado de solapamiento entre las clases en el espacio de características.

## 8. Comparación entre los resultados arrojados por los modelos de clasificación de TCH y sacarosa

In [ ]:
resumen = pd.DataFrame({
    "Objetivo": ["TCH", "Sacarosa"],
    "F1_KNN": [
        f1_score(y_test, y_pred_knn, average="macro"),
        f1_score(y_test_sac, y_pred_sac, average="macro")
    ]
})

resumen

La tabla anterior permite comparar el desempeño del modelo KNN en ambas variables objetivo. Se observa que el modelo obtiene un F1 macro de aproximadamente 0.49 para TCH y 0.50 para sacarosa.

Esta diferencia, aunque leve, sugiere que la variable sacarosa presenta una mayor facilidad de clasificación en comparación con TCH. Esto puede deberse a que los factores que influyen en la calidad del cultivo (sacarosa) están mejor representados en las variables predictoras que aquellos relacionados con el rendimiento.

En ambos casos, el desempeño del modelo se considera moderado, lo cual indica que existe un grado importante de solapamiento entre las clases, especialmente en la categoría "Medio". Este comportamiento ya había sido observado en los modelos anteriores.

En general, el modelo KNN logra capturar mejor la estructura de los datos en comparación con la regresión logística, lo que sugiere que las relaciones entre las variables no son completamente lineales y que los patrones locales juegan un papel importante en la clasificación.

## 9. Conclusiones

El modelo KNN permitió capturar patrones no lineales en los datos, ofreciendo una alternativa al modelo de regresión logística. Debido a que KNN se basa en la cercanía entre observaciones, su desempeño depende de la estructura local del conjunto de datos.

En comparación con la regresión logística, el modelo KNN presenta una mejora en el desempeño, pasando de un F1 macro cercano a 0.44 a valores cercanos a 0.49–0.50.

Esto confirma que el uso de modelos no lineales puede ser más adecuado para este tipo de problema, donde la separación entre clases no es claramente lineal.

En general, este modelo complementa el análisis realizado con regresión logística y permite una mejor comprensión del comportamiento de las variables en el proceso productivo de la caña de azúcar, pero no deja de ser un modelo poco eficiente para este problema.